In [0]:
# ===========================================
# Notebook Name:
# 00_ingest_tournament_list
#
# Purpose:
# Ingest official Pokemon Card tournament
# search API responses into Bronze.
#
# Scope:
# Events held on or after 2026-01-01
#
# API:
# https://players.pokemon-card.com/event_search
#
# Outputs:
# pokemon.bronze.tournament_list_raw
# pokemon.bronze.scrape_log (source_type='tournament_list')
#
# Bronze grain:
# One row per API response page
#
# Important:
# Bronze stores the original API response.
# The 2026 date filter is applied again when
# building the Silver tournaments table.
# ===========================================

In [0]:
from datetime import date, datetime, timezone
import hashlib
import json
import time
import uuid

import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DateType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

In [0]:
TOURNAMENT_SEARCH_API_URL = (
    "https://players.pokemon-card.com/event_search"
)

TOURNAMENT_LIST_PAGE_URL = (
    "https://players.pokemon-card.com/"
    "event/result/list"
)

TOURNAMENT_LIST_RAW_TABLE = (
    "pokemon.bronze.tournament_list_raw"
)

SCRAPE_LOG_TABLE = (
    "pokemon.bronze.scrape_log"
)

MIN_EVENT_DATE_TEXT = "2026-01-01"

MIN_EVENT_DATE = datetime.strptime(
    MIN_EVENT_DATE_TEXT,
    "%Y-%m-%d",
).date()

PAGE_SIZE = 20
MAX_PAGES = 200

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 1.0

EVENT_TYPES = [
    "3:1",
    "3:2",
    "3:7",
]

ORDER = 4
RESULT_RESIST = 1

print("API:", TOURNAMENT_SEARCH_API_URL)
print("Minimum event date:", MIN_EVENT_DATE)
print("Raw table:", TOURNAMENT_LIST_RAW_TABLE)
print("Log table:", SCRAPE_LOG_TABLE)

In [0]:
retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=1.0,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=[
        "GET",
    ],
)

http_adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "https://",
    http_adapter,
)

session.mount(
    "http://",
    http_adapter,
)

In [0]:
REQUEST_HEADERS = {
    "Accept": (
        "application/json, "
        "text/plain, */*"
    ),
    "Accept-Language": (
        "ja,en-US;q=0.9,en;q=0.8"
    ),
    "Referer": TOURNAMENT_LIST_PAGE_URL,
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/150.0.0.0 "
        "Safari/537.36"
    ),
    "X-Accept-Version": "v1",
}

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{TOURNAMENT_LIST_RAW_TABLE}
(
    run_id STRING NOT NULL,

    page_number INT NOT NULL,
    offset INT NOT NULL,
    page_size INT NOT NULL,

    request_url STRING NOT NULL,
    request_params STRING NOT NULL,

    response_status INT NOT NULL,
    api_code INT,

    payload STRING NOT NULL,
    response_hash STRING NOT NULL,

    event_count INT NOT NULL,
    eligible_event_count INT NOT NULL,

    page_min_event_date DATE,
    page_max_event_date DATE,

    contains_eligible_events BOOLEAN NOT NULL,
    all_events_before_cutoff BOOLEAN NOT NULL,

    elapsed_ms BIGINT,
    fetched_at TIMESTAMP NOT NULL,
    ingestion_date DATE NOT NULL
)
USING DELTA
COMMENT 'Raw official Pokemon Card tournament search API responses'
""")

# NOTE: pokemon.bronze.scrape_log is the unified
# fetch-attempt log shared across tournament_list /
# event_result / deck sources (see
# notebooks/00_migration/02_create_v2_tables.ipynb
# for the single-source-of-truth DDL). This notebook
# only appends to it with source_type='tournament_list'.
scrape_log_schema = spark.table(SCRAPE_LOG_TABLE).schema

print("scrape_log columns:", [f.name for f in scrape_log_schema.fields])

In [0]:
def parse_event_date_params(
    value,
):
    if value is None:
        return None

    text = str(value).strip()

    if not text:
        return None

    try:
        return datetime.strptime(
            text,
            "%Y%m%d",
        ).date()

    except ValueError:
        return None

In [0]:
assert (
    parse_event_date_params("20260607")
    == date(2026, 6, 7)
)

assert (
    parse_event_date_params(None)
    is None
)

print(
    "Validation passed: "
    "event date parser works"
)

In [0]:
def build_request_params(
    offset: int,
):
    params = [
        (
            "offset",
            offset,
        ),
        (
            "order",
            ORDER,
        ),
        (
            "result_resist",
            RESULT_RESIST,
        ),
    ]

    for event_type in EVENT_TYPES:
        params.append(
            (
                "event_type[]",
                event_type,
            )
        )

    return params

In [0]:
print(
    build_request_params(0)
)

In [0]:
def serialize_payload(
    payload_object,
) -> str:
    return json.dumps(
        payload_object,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )

In [0]:
def calculate_sha256(
    value: str,
) -> str:
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()

In [0]:
def fetch_tournament_page(
    offset: int,
):
    params = build_request_params(
        offset
    )

    started_at = time.perf_counter()

    response = session.get(
        TOURNAMENT_SEARCH_API_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    elapsed_ms = int(
        (
            time.perf_counter()
            - started_at
        )
        * 1000
    )

    response.raise_for_status()

    content_type = (
        response.headers
        .get(
            "Content-Type",
            "",
        )
        .lower()
    )

    if (
        "application/json"
        not in content_type
    ):
        raise ValueError(
            "Tournament search API did "
            "not return JSON. "
            f"status={response.status_code}, "
            f"content_type={content_type}, "
            f"body={response.text[:500]}"
        )

    try:
        payload_object = response.json()

    except ValueError as error:
        raise ValueError(
            "Tournament search response "
            "could not be parsed as JSON. "
            f"body={response.text[:500]}"
        ) from error

    api_code = payload_object.get(
        "code"
    )

    if (
        api_code is not None
        and int(api_code) != 200
    ):
        raise ValueError(
            "Tournament search API "
            "returned a non-success code. "
            f"api_code={api_code}"
        )

    events = payload_object.get(
        "event",
        [],
    )

    if events is None:
        events = []

    if not isinstance(
        events,
        list,
    ):
        raise ValueError(
            "Expected payload['event'] "
            "to be a list. "
            f"actual_type={type(events)}"
        )

    payload = serialize_payload(
        payload_object
    )

    response_hash = calculate_sha256(
        payload
    )

    return {
        "request_url": response.url,
        "request_params": json.dumps(
            params,
            ensure_ascii=False,
        ),
        "response_status": (
            response.status_code
        ),
        "api_code": (
            int(api_code)
            if api_code is not None
            else None
        ),
        "payload_object": payload_object,
        "payload": payload,
        "response_hash": response_hash,
        "events": events,
        "elapsed_ms": elapsed_ms,
    }

In [0]:
sample_result = (
    fetch_tournament_page(
        offset=0
    )
)

print(
    "Status:",
    sample_result[
        "response_status"
    ],
)

print(
    "API code:",
    sample_result[
        "api_code"
    ],
)

print(
    "Event count:",
    len(
        sample_result["events"]
    ),
)

print(
    "Response hash:",
    sample_result[
        "response_hash"
    ],
)

In [0]:
if sample_result["events"]:
    print(
        json.dumps(
            sample_result[
                "events"
            ][0],
            ensure_ascii=False,
            indent=2,
        )
    )

In [0]:
sample_event_dates = [
    parse_event_date_params(
        event.get(
            "event_date_params"
        )
    )
    for event in (
        sample_result["events"]
    )
]

sample_event_dates = [
    event_date
    for event_date
    in sample_event_dates
    if event_date is not None
]

print(
    "Sample page date range:",
    (
        min(sample_event_dates)
        if sample_event_dates
        else None
    ),
    "-",
    (
        max(sample_event_dates)
        if sample_event_dates
        else None
    ),
)

In [0]:
is_descending = all(
    sample_event_dates[index]
    >= sample_event_dates[index + 1]
    for index in range(
        len(sample_event_dates) - 1
    )
)

print(
    "Dates are descending:",
    is_descending,
)

In [0]:
existing_response_hashes = {
    row["response_hash"]
    for row in (
        spark.table(
            TOURNAMENT_LIST_RAW_TABLE
        )
        .select(
            "response_hash"
        )
        .distinct()
        .collect()
    )
}

print(
    "Existing response hashes:",
    len(existing_response_hashes),
)

In [0]:
run_id = str(
    uuid.uuid4()
)

run_started_at = datetime.now(
    timezone.utc
)

raw_rows = []

seen_hashes_in_run = set()
eligible_events_by_id = {}

requested_page_count = 0
successful_page_count = 0
inserted_page_count = 0
skipped_page_count = 0
error_page_count = 0

total_api_event_count = 0
eligible_event_count = 0

last_offset = None
stop_reason = None
run_error_message = None

print(
    "Run ID:",
    run_id,
)

In [0]:
try:
    for page_number in range(
        1,
        MAX_PAGES + 1,
    ):
        offset = (
            page_number - 1
        ) * PAGE_SIZE

        last_offset = offset
        requested_page_count += 1

        fetched_at = datetime.now(
            timezone.utc
        )

        try:
            result = fetch_tournament_page(
                offset=offset
            )

            successful_page_count += 1

        except Exception:
            error_page_count += 1
            raise

        page_events = result[
            "events"
        ]

        page_event_records = []

        for event in page_events:
            event_date = (
                parse_event_date_params(
                    event.get(
                        "event_date_params"
                    )
                )
            )

            page_event_records.append({
                "event": event,
                "event_date": event_date,
            })

        parsed_page_dates = [
            item["event_date"]
            for item
            in page_event_records
            if (
                item["event_date"]
                is not None
            )
        ]

        eligible_page_records = [
            item
            for item
            in page_event_records
            if (
                item["event_date"]
                is not None
                and item["event_date"]
                >= MIN_EVENT_DATE
            )
        ]

        page_event_count = len(
            page_events
        )

        eligible_page_event_count = len(
            eligible_page_records
        )

        total_api_event_count += (
            page_event_count
        )

        eligible_event_count += (
            eligible_page_event_count
        )

        for item in (
            eligible_page_records
        ):
            event = item["event"]
            event_id = event.get("id")

            if event_id is None:
                continue

            eligible_events_by_id[
                str(event_id)
            ] = event

        page_min_event_date = (
            min(parsed_page_dates)
            if parsed_page_dates
            else None
        )

        page_max_event_date = (
            max(parsed_page_dates)
            if parsed_page_dates
            else None
        )

        contains_eligible_events = (
            eligible_page_event_count > 0
        )

        all_events_before_cutoff = (
            bool(parsed_page_dates)
            and all(
                event_date
                < MIN_EVENT_DATE
                for event_date
                in parsed_page_dates
            )
        )

        response_hash = result[
            "response_hash"
        ]

        is_new_response = (
            response_hash
            not in existing_response_hashes
            and response_hash
            not in seen_hashes_in_run
        )

        if is_new_response:
            raw_rows.append({
                "run_id": run_id,
                "page_number": (
                    page_number
                ),
                "offset": offset,
                "page_size": PAGE_SIZE,
                "request_url": result[
                    "request_url"
                ],
                "request_params": result[
                    "request_params"
                ],
                "response_status": result[
                    "response_status"
                ],
                "api_code": result[
                    "api_code"
                ],
                "payload": result[
                    "payload"
                ],
                "response_hash": (
                    response_hash
                ),
                "event_count": (
                    page_event_count
                ),
                "eligible_event_count": (
                    eligible_page_event_count
                ),
                "page_min_event_date": (
                    page_min_event_date
                ),
                "page_max_event_date": (
                    page_max_event_date
                ),
                "contains_eligible_events": (
                    contains_eligible_events
                ),
                "all_events_before_cutoff": (
                    all_events_before_cutoff
                ),
                "elapsed_ms": result[
                    "elapsed_ms"
                ],
                "fetched_at": fetched_at,
                "ingestion_date": (
                    fetched_at.date()
                ),
            })

            seen_hashes_in_run.add(
                response_hash
            )

            inserted_page_count += 1

            row_status = "new"

        else:
            skipped_page_count += 1
            row_status = (
                "skipped_unchanged"
            )

        print(
            f"page={page_number}, "
            f"offset={offset}, "
            f"events={page_event_count}, "
            f"eligible={eligible_page_event_count}, "
            f"date_range="
            f"{page_min_event_date}"
            f" - "
            f"{page_max_event_date}, "
            f"status={row_status}"
        )

        if page_event_count == 0:
            stop_reason = (
                "empty_event_page"
            )

            print(
                "Stop: API returned "
                "an empty event page."
            )

            break

        if all_events_before_cutoff:
            stop_reason = (
                "all_page_events_before_"
                "2026_01_01"
            )

            print(
                "Stop: all events on "
                "this page are before "
                f"{MIN_EVENT_DATE_TEXT}."
            )

            break

        if page_event_count < PAGE_SIZE:
            stop_reason = (
                "last_partial_page"
            )

            print(
                "Stop: API returned "
                "fewer than PAGE_SIZE events."
            )

            break

        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )

    else:
        stop_reason = (
            "max_pages_reached"
        )

except Exception as error:
    stop_reason = (
        "error"
    )

    run_error_message = (
        f"{type(error).__name__}: "
        f"{str(error)}"
    )[:4000]

    print(
        "Ingestion failed:",
        run_error_message,
    )

In [0]:
raw_schema = StructType([
    StructField(
        "run_id",
        StringType(),
        False,
    ),
    StructField(
        "page_number",
        IntegerType(),
        False,
    ),
    StructField(
        "offset",
        IntegerType(),
        False,
    ),
    StructField(
        "page_size",
        IntegerType(),
        False,
    ),
    StructField(
        "request_url",
        StringType(),
        False,
    ),
    StructField(
        "request_params",
        StringType(),
        False,
    ),
    StructField(
        "response_status",
        IntegerType(),
        False,
    ),
    StructField(
        "api_code",
        IntegerType(),
        True,
    ),
    StructField(
        "payload",
        StringType(),
        False,
    ),
    StructField(
        "response_hash",
        StringType(),
        False,
    ),
    StructField(
        "event_count",
        IntegerType(),
        False,
    ),
    StructField(
        "eligible_event_count",
        IntegerType(),
        False,
    ),
    StructField(
        "page_min_event_date",
        DateType(),
        True,
    ),
    StructField(
        "page_max_event_date",
        DateType(),
        True,
    ),
    StructField(
        "contains_eligible_events",
        BooleanType(),
        False,
    ),
    StructField(
        "all_events_before_cutoff",
        BooleanType(),
        False,
    ),
    StructField(
        "elapsed_ms",
        LongType(),
        True,
    ),
    StructField(
        "fetched_at",
        TimestampType(),
        False,
    ),
    StructField(
        "ingestion_date",
        DateType(),
        False,
    ),
])

In [0]:
if raw_rows:
    tournament_list_raw_df = (
        spark.createDataFrame(
            raw_rows,
            schema=raw_schema,
        )
    )

    (
        tournament_list_raw_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            TOURNAMENT_LIST_RAW_TABLE
        )
    )

    print(
        "Inserted Bronze pages:",
        len(raw_rows),
    )

else:
    print(
        "No new Bronze pages "
        "to insert."
    )

In [0]:
run_finished_at = datetime.now(
    timezone.utc
)

run_elapsed_ms = int(
    (
        run_finished_at
        - run_started_at
    ).total_seconds()
    * 1000
)

unique_eligible_event_count = len(
    eligible_events_by_id
)

if run_error_message:
    run_status = "error"

elif inserted_page_count == 0:
    run_status = "success_no_change"

else:
    run_status = "success"

In [0]:
# -------------------------------------------
# Write a single unified scrape_log row for this
# run (source_type='tournament_list'). Per-page
# detail (requested/successful/skipped page counts,
# stop_reason, etc.) is preserved in this notebook's
# own stdout/print output, not in scrape_log, since
# scrape_log's grain is one row per fetch attempt
# shared across all Bronze sources.
# -------------------------------------------
log_row = {
    "run_id": run_id,
    "source_type": "tournament_list",
    "source_id": run_id,
    "request_url": TOURNAMENT_SEARCH_API_URL,
    "http_status": None,
    "status": run_status,
    "elapsed_ms": run_elapsed_ms,
    "records_found": total_api_event_count,
    "error_type": (
        "ingestion_error"
        if run_error_message
        else None
    ),
    "error_message": run_error_message,
    "scraped_at": run_finished_at,
    "ingestion_date": run_finished_at.date(),
}

scrape_log_df = spark.createDataFrame(
    [log_row],
    schema=scrape_log_schema,
)

(
    scrape_log_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        SCRAPE_LOG_TABLE
    )
)

print(
    "Tournament list run log saved to",
    SCRAPE_LOG_TABLE,
)

In [0]:
print("=" * 60)
print("Tournament list ingestion completed")
print("=" * 60)

print(
    "Run ID:",
    run_id,
)

print(
    "Status:",
    run_status,
)

print(
    "Stop reason:",
    stop_reason,
)

print(
    "Requested pages:",
    requested_page_count,
)

print(
    "Successful pages:",
    successful_page_count,
)

print(
    "Inserted pages:",
    inserted_page_count,
)

print(
    "Skipped pages:",
    skipped_page_count,
)

print(
    "API event rows:",
    total_api_event_count,
)

print(
    "Eligible event rows:",
    eligible_event_count,
)

print(
    "Unique eligible events:",
    unique_eligible_event_count,
)

print(
    "Elapsed milliseconds:",
    run_elapsed_ms,
)

In [0]:
display(
    spark.table(
        TOURNAMENT_LIST_RAW_TABLE
    )
    .filter(
        F.col("run_id") == run_id
    )
    .select(
        "page_number",
        "offset",
        "event_count",
        "eligible_event_count",
        "page_min_event_date",
        "page_max_event_date",
        "contains_eligible_events",
        "all_events_before_cutoff",
        "response_hash",
        "elapsed_ms",
        "fetched_at",
    )
    .orderBy(
        "page_number"
    )
)

In [0]:
display(
    spark.table(
        SCRAPE_LOG_TABLE
    )
    .filter(
        F.col("source_type") == "tournament_list"
    )
    .orderBy(
        F.col(
            "scraped_at"
        ).desc()
    )
    .limit(20)
)

In [0]:
duplicate_hash_df = (
    spark.table(
        TOURNAMENT_LIST_RAW_TABLE
    )
    .groupBy(
        "response_hash"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_hash_count = (
    duplicate_hash_df.count()
)

if duplicate_hash_count > 0:
    display(
        duplicate_hash_df
    )

    raise ValueError(
        f"{duplicate_hash_count} "
        "duplicate response hashes detected"
    )

print(
    "Validation passed: "
    "no duplicate response hashes"
)

In [0]:
invalid_status_df = (
    spark.table(
        TOURNAMENT_LIST_RAW_TABLE
    )
    .filter(
        F.col(
            "response_status"
        ) != 200
    )
)

invalid_status_count = (
    invalid_status_df.count()
)

if invalid_status_count > 0:
    display(
        invalid_status_df
    )

    raise ValueError(
        f"{invalid_status_count} "
        "non-200 Bronze responses detected"
    )

print(
    "Validation passed: "
    "all Bronze responses are HTTP 200"
)

In [0]:
if run_error_message:
    raise RuntimeError(
        "Tournament list ingestion failed. "
        f"run_id={run_id}, "
        f"error={run_error_message}"
    )